# 32. Target encoding the pairs

**One variable against ledger row 38** (`xgb_te`, CV 0.967099): the feature set. Same
learner, same folds, same seed, same budget, same `SMOOTH`, same `N_INNER`, same 1-way
encoder. Only crossed columns are added.

## Where this came from

`31_interaction_diagnostic.ipynb` measured what every pair of columns knows that the two
columns separately do not, and refuted the clause` that had feature
engineering closed since 2026-08-11. Nine of the 66 pairs clear a within-target-shuffled
control whose band is centred on zero with a maximum of +0.001739. The strongest,
`daily_screen_time_hours` x `work_study_hours`, lifts single-feature AUC by **+0.016446**
over the best additive combination of its two columns.

That prices the idea. It does not settle it, and the reason is row 19.

## Why this might still be worth nothing, stated before the run

**A gradient boosted tree already models interactions.** That is most of what depth is
for, and this configuration runs at depth 6 on 36 features, so every 2-way interaction
here is already reachable. `31` measured the lift over an **additive** baseline, not over
a **tree**. Those are different baselines and only one of them is the thing being
submitted.

The decimal lattice, ledger row 19, is exactly this shape and it is the closest
precedent in the repo: an effect verified in the data, large, real, and worth
**-0.000132** in the model, because the model already had it.

## Why it might not be nothing

This repo argues 1-way target encoding paid because a level "arrives as one number the
tree can split on in a single cut, where before they cost many". A 2D cell costs a
depth-6 tree substantially more than one cut: it has to spend depth carving the
rectangle, and depth spent there is depth not spent elsewhere. Handing it the cell
pre-computed is the identical trick that worked the first time, one dimension up.

So the two arguments point in opposite directions and both are written here before the
run. **My prediction: a real gain, smaller than 1-way encoding's +0.003312 and larger
than the +0.00010 gate.** Recording it so it can be wrong.

## Three arms, and why the leak-free one is the headline

| arm | features | what it is |
|---|---|---|
| `base` | 36 | row 38 refit in this kernel, the reproduction check |
| `top9` | 54 | the nine pairs that cleared `31`'s control |
| `all66` | 168 | every pair, no selection at all |

**`all66` is the headline because it involves no selection.** `top9` picks its pairs
using scores measured on fold 0, so fold 0's contribution to `top9`'s CV is
optimistically biased. That is selection on the validation set, the error this repo
refused to commit in row 45 and will not commit here by omission either. The notebook
therefore also reports `top9` over **folds 1 to 4 only**, where the selection never
looked, and the gap between the two numbers is the size of the bias.

`all66` cannot have that problem. If it pays, the result is clean.

## The gate, pre-registered

Reused from `29` and `30` so all three are comparable, judged against the `base` arm
refit in this same kernel rather than against the recorded ledger number.

| verdict | condition |
|---|---|
| `carry to a second seed` | wins >= 4/5, mean > 2 x paired sd, and mean >= +0.00010 |
| `parity` | paired-significant but under +0.00010 |
| `null` | anything else |

## Binning, and where the edges come from

Crossing needs levels, and six of the columns are continuous. Quantile edges are
computed **inside each outer fold, from the training portion only**. Bin edges use no
target information at all, so computing them on everything would not be a leak in the
strict sense, but doing it per fold costs nothing and removes the need for that
paragraph to be true.


In [ ]:
# One flag. The full run always goes top to bottom on Kaggle.
SMOKE = True

SEED = 42
N_INNER = 5
SMOOTH = 10.0

# Row 38's configuration, held.
LR = 0.05
N_EST = 2000
BENCH_EST = 200
PROBE_FOLD = 0
MAX_DEPTH = 6
N_JOBS = -1

# Crossing. 12 quantile levels is 31's setting, so the pairs measured there are the
# pairs built here.
N_BINS = 12
RAW_AS_LEVELS = ("age",)

# The nine pairs from 31 that cleared the shuffled control's maximum, in its order.
# Written out rather than read from artifacts/, because this notebook has to run on
# Kaggle where that directory does not exist.
TOP_PAIRS = [
    ("daily_screen_time_hours", "work_study_hours"),
    ("daily_screen_time_hours", "gaming_hours"),
    ("daily_screen_time_hours", "social_media_hours"),
    ("work_study_hours", "weekend_screen_time"),
    ("social_media_hours", "gaming_hours"),
    ("gaming_hours", "weekend_screen_time"),
    ("social_media_hours", "work_study_hours"),
    ("daily_screen_time_hours", "weekend_screen_time"),
    ("social_media_hours", "weekend_screen_time"),
]

ARMS = ["base", "top9", "all66"]

BASELINE_NAME = "xgb_te"
BASELINE_CV = 0.967099
EXPECTED_FOLD_SHA = "ec282b0968059676"

EXPECTED_LEAK2 = 8.1e-05
EXPECTED_PRIOR_SHIFT = 1.3e-04

GATE_FLOOR = 1.0e-04

print(f"SMOKE = {SMOKE}   arms {ARMS}   {len(TOP_PAIRS)} selected pairs")


## Stage 1. Data, folds, leak checklist

The fold checksum is the only thing standing between an out-of-fold vector that
blends and one that is silently misaligned, so it is checked before anything trains
rather than after.

In [ ]:
import itertools
import ast
import gc
import hashlib
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold

# Runs here or on Kaggle. Both are found by name rather than by assuming a shape.
KAG = Path("/kaggle/input")
ON_KAGGLE = KAG.exists()
LOCAL = next((b for b in [Path.cwd(), *Path.cwd().parents]
              if (b / "data" / "raw" / "train.csv").exists()), None)


def locate(name):
    if ON_KAGGLE:
        hits = sorted(KAG.rglob(name))
        if hits:
            return hits[0]
    if LOCAL is not None:
        for d in ("data/raw", "artifacts/oof", "notebooks", "submissions"):
            p = LOCAL / d / name
            if p.exists():
                return p
    raise FileNotFoundError(name)


OUT = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts" / "oof"
SUB = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "submissions"
print(f"running {'on Kaggle' if ON_KAGGLE else 'locally'}, writing to {OUT}")

train_full = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
TARGET = "addicted_label"
CAT = ["gender", "stress_level", "academic_work_impact"]
COLS = [c for c in train_full.columns if c not in ("id", TARGET)]

# Leak checklist, re-run rather than ticked by inspection. `id` is a contiguous row
# index that separates train from test perfectly, so it is a guaranteed leak if it
# ever reaches the model.
checks = {
    "id is not a feature": "id" not in COLS,
    "target is not a feature": TARGET not in COLS,
    "train and test ids do not overlap":
        not (set(train_full["id"]) & set(test["id"])),
    "train and test feature lists match":
        COLS == [c for c in test.columns if c != "id"],
}
for name, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")
LEAK_OK = all(checks.values())

# ROW_IDX maps this run's rows back into the saved member vectors.
if SMOKE:
    ROW_IDX = np.sort(train_full.sample(20000, random_state=0).index.to_numpy())
    train = train_full.loc[ROW_IDX].reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
    N_EST, BENCH_EST = 200, 50
    # Measured 2026-08-19 and written up: at 16,000 rows this machine
    # runs 101x slower at n_jobs=-1 than at n_jobs=1, monotone in the thread count.
    # N_JOBS above is chosen to match row 17 on Kaggle at 691,369 rows, where it is
    # right. A smoke run produces no ledger number, so overriding it here costs
    # nothing and is the difference between two minutes and giving up on the check.
    N_JOBS = 1
else:
    ROW_IDX = np.arange(len(train_full))
    train = train_full

y = train[TARGET].to_numpy()
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i

sha = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
ALIGNED = sha == EXPECTED_FOLD_SHA
print()
print(f"rows {len(train):,}   target rate {y.mean():.6f}")
print(f"fold sizes {np.bincount(folds).tolist()}")
print(f"fold sha {sha}  expected {EXPECTED_FOLD_SHA}")
if SMOKE:
    print("SMOKE: subsampled, so the sha is EXPECTED to differ. Not a check.")
else:
    print("fold alignment: VERIFIED" if ALIGNED else
          "fold alignment: MISMATCH - the OOF from this run is not blendable")

X = train[COLS].copy()
X_test = test[COLS].copy()
for c in CAT:
    X[c] = X[c].astype("category")
    X_test[c] = X_test[c].astype("category")

## Stage 2. The encoder

Copied from `13_target_encoding.ipynb` so the feature set is row 17's feature set. A
copy is a provenance risk under the notebook layout, so it is checked rather than
asserted: the cell below parses the encoder out of `13`, normalises both versions
through `ast.unparse`, and compares checksums. Expected fingerprint
`0642e41750ef8bab`, the same value rows 26 and 33 recorded.

In [ ]:
def _stats(levels, yy, prior):
    """Smoothed target mean and level frequency, fit only on the rows given."""
    df = pd.DataFrame({"v": levels, "y": yy})
    g = df.groupby("v", dropna=False, observed=True)["y"].agg(["sum", "count"])
    mean = (g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH)
    return mean, g["count"] / len(df)


def _apply(levels, mean, freq, prior):
    s = pd.Series(levels)
    m = s.map(mean).to_numpy(dtype=np.float64)
    f = s.map(freq).to_numpy(dtype=np.float64)
    # A level unseen while fitting falls back to the prior and zero mass.
    return np.nan_to_num(m, nan=prior), np.nan_to_num(f, nan=0.0)


def encode_fold(Xf, yy, tr, va, Xt=None, seed=SEED):
    """Encodings for ONE outer fold: (train, valid, test)."""
    prior = float(yy[tr].mean())
    e_tr, e_va, e_te = {}, {}, {}
    splits = list(KFold(N_INNER, shuffle=True, random_state=seed).split(tr))

    for c in COLS:
        lv = Xf[c].to_numpy()
        # Fit once on the whole training portion, for validation and test rows.
        mean, freq = _stats(lv[tr], yy[tr], prior)
        e_va[f"te_{c}"], e_va[f"fq_{c}"] = _apply(lv[va], mean, freq, prior)
        if Xt is not None:
            e_te[f"te_{c}"], e_te[f"fq_{c}"] = _apply(Xt[c].to_numpy(), mean,
                                                      freq, prior)
        # Training rows get inner out-of-fold values.
        tm, tf = np.empty(len(tr)), np.empty(len(tr))
        for itr, iva in splits:
            im, if_ = _stats(lv[tr[itr]], yy[tr[itr]], prior)
            tm[iva], tf[iva] = _apply(lv[tr[iva]], im, if_, prior)
        e_tr[f"te_{c}"], e_tr[f"fq_{c}"] = tm, tf

    return (pd.DataFrame(e_tr), pd.DataFrame(e_va),
            pd.DataFrame(e_te) if Xt is not None else None)


def build(Xf, yy, tr, va, Xt=None, seed=SEED):
    d_tr, d_va, d_te = encode_fold(Xf, yy, tr, va, Xt, seed)
    Xtr = pd.concat([Xf.iloc[tr].reset_index(drop=True), d_tr], axis=1)
    Xva = pd.concat([Xf.iloc[va].reset_index(drop=True), d_va], axis=1)
    Xte = None if Xt is None else pd.concat(
        [Xt.reset_index(drop=True), d_te], axis=1)
    return Xtr, Xva, Xte


ENCODER_FNS = ("_stats", "_apply", "encode_fold", "build")


def fingerprint(src):
    """Semantic checksum of the encoder functions inside a block of source."""
    body = ast.parse(src).body
    parts = [ast.unparse(n) for n in body
             if isinstance(n, ast.FunctionDef) and n.name in ENCODER_FNS]
    if len(parts) != len(ENCODER_FNS):
        return None
    return hashlib.sha256("\n".join(parts).encode()).hexdigest()[:16]


import inspect

mine = fingerprint("\n".join(inspect.getsource(f)
                             for f in (_stats, _apply, encode_fold, build)))

theirs = None
try:
    src13 = locate("13_target_encoding.ipynb")
except FileNotFoundError:
    src13 = None
if src13 is not None:
    import json as _json
    for c in _json.loads(src13.read_text(encoding="utf-8"))["cells"]:
        if c["cell_type"] == "code" and "def encode_fold" in "".join(c["source"]):
            theirs = fingerprint("".join(c["source"]))
            break

ENCODER_MATCH = mine is not None and mine == theirs
print(f"encoder fingerprint here        : {mine}")
print(f"encoder fingerprint in 13       : {theirs}")
print(f"rows 26 and 33 recorded         : 0642e41750ef8bab")
print("encoder: IDENTICAL to row 17's" if ENCODER_MATCH else
      "encoder: DIFFERS from 13 (or 13 not found) - this is NOT one variable")
print()
print(f"{len(COLS)} raw columns -> {len(COLS) * 3} features after encoding")

### The leak checks, by execution

The same three checks `13` ran, on the same encoder, so their numbers are directly
comparable to the ones recorded earlier. Read all three together: the first two must be
about zero, the third must be large. Without the third, an encoder that ignored the
target entirely would pass the first two and look clean.

In [ ]:
_tr = np.where(folds != 0)[0]
_va = np.where(folds == 0)[0]
d_tr0, d_va0, _ = encode_fold(X, y, _tr, _va)

# 1. A validation row's own target must never reach its own encoding.
y1 = y.copy()
y1[_va] = 1 - y1[_va]
_, d_va1, _ = encode_fold(X, y1, _tr, _va)
leak1 = max(np.abs(d_va0[f"te_{c}"] - d_va1[f"te_{c}"]).max() for c in COLS)

# 2. A training row's own target must never reach its own inner encoding. A small
# residual is expected and is not a leak: `prior` is the training-portion mean.
_, iva0 = list(KFold(N_INNER, shuffle=True, random_state=SEED).split(_tr))[0]
pick = iva0[:200]
y2 = y.copy()
y2[_tr[pick]] = 1 - y2[_tr[pick]]
d_tr2, _, _ = encode_fold(X, y2, _tr, _va)
leak2 = max(np.abs(d_tr0[f"te_{c}"].to_numpy()[pick]
                   - d_tr2[f"te_{c}"].to_numpy()[pick]).max() for c in COLS)
prior_shift = abs(float(y2[_tr].mean()) - float(y[_tr].mean()))

# 3. The encoding MUST move when targets it is allowed to see change.
y3 = y.copy()
y3[_tr] = 1 - y3[_tr]
_, d_va3, _ = encode_fold(X, y3, _tr, _va)
live = max(np.abs(d_va0[f"te_{c}"] - d_va3[f"te_{c}"]).max() for c in COLS)

print(f"1. flip all validation targets -> change in their encoding: {leak1:.3e}")
print(f"2. flip 200 training rows -> change in their own encoding:  {leak2:.3e}")
print(f"   prior moved {prior_shift:.3e}, and these should track each other")
print(f"3. flip all training targets -> change in val encoding:     {live:.3e}")

CLEAN = leak1 == 0 and leak2 < 10 * max(prior_shift, 1e-9) and live > 0.1
print()
print("LEAK CHECKS: PASS" if CLEAN else "LEAK CHECKS: FAILED - do not log this run")
if not SMOKE:
    print(f"13 recorded leak2 {EXPECTED_LEAK2:.1e} against a prior shift of "
          f"{EXPECTED_PRIOR_SHIFT:.1e}; this run gives {leak2:.1e} and "
          f"{prior_shift:.1e}")

del d_tr0, d_va0, d_va1, d_tr2, d_va3, y1, y2, y3
gc.collect()

## Stage 2b. The crossed encoder

`_stats` and `_apply` are reused unchanged, so the smoothing and the unseen-level
fallback are the same code the 1-way encoding uses. `encode_fold` and `build` are also
untouched, which is what keeps the fingerprint check against `13` meaningful: this
notebook adds an encoder, it does not modify one.

The nesting is identical in structure. Validation and test rows get a cross fit once on
the whole training portion; training rows get inner out-of-fold values from the same
`KFold(N_INNER)` split. Anything less would put a row's own target into its own crossed
feature, which is the one mistake that would make every number here meaningless.


In [ ]:
ALL_PAIRS = list(itertools.combinations(COLS, 2))
print(f"{len(ALL_PAIRS)} pairs from {len(COLS)} columns")


def fold_levels(tr, Xt=None):
    """Integer levels for the train frame and, if given, the test frame.

    Both frames are coded together on purpose. Factorising them separately would give
    the same category a different integer in each, and every test-row cross would then
    address the wrong cell. Numeric bin edges come from the training rows only, and
    edges use no target information, so this is binning rather than leakage.

    Levels are shifted to start at 0 across both frames so that the pair key below can
    use one multiplier for both. Deriving the offset per array is the same bug in a
    different place.
    """
    lv, lvt = {}, {}
    for c in COLS:
        s = X[c]
        # StringDtype, not object, in this pandas. Ask whether it is numeric.
        if (c in RAW_AS_LEVELS or c in CAT
                or not pd.api.types.is_numeric_dtype(s)):
            joined = pd.concat([s, Xt[c]], ignore_index=True) if Xt is not None else s
            codes = pd.Categorical(joined).codes.astype(np.int64)  # NaN becomes -1
            lv[c] = codes[:len(s)]
            if Xt is not None:
                lvt[c] = codes[len(s):]
        else:
            v = s.to_numpy(dtype=np.float64)
            edges = np.unique(np.nanquantile(v[tr], np.linspace(0, 1, N_BINS + 1)))
            lv[c] = np.where(np.isnan(v), -1,
                             np.digitize(v, edges[1:-1], right=False)).astype(np.int64)
            if Xt is not None:
                vt = Xt[c].to_numpy(dtype=np.float64)
                lvt[c] = np.where(np.isnan(vt), -1,
                                  np.digitize(vt, edges[1:-1],
                                              right=False)).astype(np.int64)
        lo = min(lv[c].min(), lvt[c].min()) if Xt is not None else lv[c].min()
        lv[c] = lv[c] - lo
        if Xt is not None:
            lvt[c] = lvt[c] - lo
    return lv, lvt


def encode_pairs(yy, tr, va, pairs, lv, lvt=None, seed=SEED):
    """Crossed encodings for ONE outer fold, nested exactly as encode_fold is."""
    prior = float(yy[tr].mean())
    splits = list(KFold(N_INNER, shuffle=True, random_state=seed).split(tr))
    e_tr, e_va, e_te = {}, {}, {}

    for ca, cb in pairs:
        name = f"{ca}__{cb}"
        # One multiplier, taken across both frames, so a train key and a test key for
        # the same pair of bins are the same integer.
        kb = max(lv[cb].max(), lvt[cb].max() if lvt else 0) + 1
        key = lv[ca] * kb + lv[cb]

        # Validation and test read an encoder fit once on the whole training portion.
        mean, freq = _stats(key[tr], yy[tr], prior)
        m, f = _apply(key[va], mean, freq, prior)
        e_va[f"xte_{name}"] = m.astype(np.float32)
        e_va[f"xfq_{name}"] = f.astype(np.float32)
        if lvt:
            kt = lvt[ca] * kb + lvt[cb]
            m, f = _apply(kt, mean, freq, prior)
            e_te[f"xte_{name}"] = m.astype(np.float32)
            e_te[f"xfq_{name}"] = f.astype(np.float32)

        # Training rows read inner out-of-fold values.
        tm = np.empty(len(tr), np.float32)
        tf = np.empty(len(tr), np.float32)
        for itr, iva in splits:
            im, if_ = _stats(key[tr[itr]], yy[tr[itr]], prior)
            a, b = _apply(key[tr[iva]], im, if_, prior)
            tm[iva], tf[iva] = a, b
        e_tr[f"xte_{name}"], e_tr[f"xfq_{name}"] = tm, tf

    return (pd.DataFrame(e_tr), pd.DataFrame(e_va),
            pd.DataFrame(e_te) if lvt else None)


def arm_cols(arm, frame):
    """Columns for one arm. The crossed block is built once and sliced, not rebuilt."""
    if arm == "base":
        return [c for c in frame.columns if not c.startswith(("xte_", "xfq_"))]
    if arm == "all66":
        return list(frame.columns)
    keep = {f"{p}_{ca}__{cb}" for ca, cb in TOP_PAIRS for p in ("xte", "xfq")}
    return [c for c in frame.columns
            if not c.startswith(("xte_", "xfq_")) or c in keep]


### The leak check the crossed encoder needs on its own

The three checks above are `13`'s and they test `13`'s encoder. The crossed encoder is
new code and gets its own, run the same way: by execution rather than by reading it.

A crossed cell is smaller than either of its columns' cells, so a leak here would be
**larger** than a leak in the 1-way encoder, not smaller. This is the check that most
needs to exist in this notebook.


In [ ]:
_tr = np.where(folds != 0)[0]
_va = np.where(folds == 0)[0]
_probe = ALL_PAIRS[:6]
_lv, _ = fold_levels(_tr)

d_tr0, d_va0, _ = encode_pairs(y, _tr, _va, _probe, _lv)

# 1. A validation row's own target must never reach its own crossed encoding.
y1 = y.copy()
y1[_va] = 1 - y1[_va]
_, d_va1, _ = encode_pairs(y1, _tr, _va, _probe, _lv)
xleak1 = max(np.abs(d_va0[c] - d_va1[c]).max()
             for c in d_va0.columns if c.startswith("xte_"))

# 2. A training row's own target must never reach its own inner crossed encoding.
_, iva0 = list(KFold(N_INNER, shuffle=True, random_state=SEED).split(_tr))[0]
pick = iva0[:200]
y2 = y.copy()
y2[_tr[pick]] = 1 - y2[_tr[pick]]
d_tr2, _, _ = encode_pairs(y2, _tr, _va, _probe, _lv)
xleak2 = max(np.abs(d_tr0[c].to_numpy()[pick] - d_tr2[c].to_numpy()[pick]).max()
             for c in d_tr0.columns if c.startswith("xte_"))
xprior = abs(float(y2[_tr].mean()) - float(y[_tr].mean()))

# 3. It MUST move when targets it is allowed to see change.
y3 = y.copy()
y3[_tr] = 1 - y3[_tr]
_, d_va3, _ = encode_pairs(y3, _tr, _va, _probe, _lv)
xlive = max(np.abs(d_va0[c] - d_va3[c]).max()
            for c in d_va0.columns if c.startswith("xte_"))

print(f"1. flip all validation targets -> change in their crossed encoding: {xleak1:.3e}")
print(f"2. flip 200 training rows -> change in their own crossed encoding:  {xleak2:.3e}")
print(f"   prior moved {xprior:.3e}, and these should track each other")
print(f"3. flip all training targets -> change in val crossed encoding:     {xlive:.3e}")

XCLEAN = xleak1 == 0 and xleak2 < 10 * max(xprior, 1e-9) and xlive > 0.1
print()
print("CROSSED LEAK CHECKS: PASS" if XCLEAN else
      "CROSSED LEAK CHECKS: FAILED - do not log this run")

del d_tr0, d_va0, d_va1, d_tr2, d_va3, y1, y2, y3
gc.collect()


## Stage 4. The three arms

The crossed block is built once per outer fold and sliced per arm, so `all66` and `top9`
share the encoder work and `base` simply drops the block. That keeps the arms paired on
identical encodings rather than on two separate builds of nominally the same thing.


In [ ]:
import xgboost as xgb


def make(n_est):
    return xgb.XGBClassifier(
        objective="binary:logistic", eval_metric="auc",
        tree_method="hist", enable_categorical=True,
        learning_rate=LR, n_estimators=n_est, max_depth=MAX_DEPTH,
        subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, n_jobs=N_JOBS, verbosity=0,
    )


def hhmm(s):
    return f"{int(s // 60)}m {int(s % 60):02d}s"


LOG = (Path("/kaggle/working") if ON_KAGGLE
       else LOCAL / "artifacts" / "logs") / "32_pair_encoding.log"
LOG.parent.mkdir(parents=True, exist_ok=True)


def note(msg):
    print(msg)
    with LOG.open("a", encoding="utf-8") as fh:
        print(f"{time.strftime('%H:%M:%S')}  {msg}", file=fh, flush=True)


note(f"=== run start, SMOKE={SMOKE}, arms={ARMS} ===")

oof = {a: np.zeros(len(train)) for a in ARMS}
test_pred = {a: np.zeros(len(test)) for a in ARMS}
per_fold = {a: [] for a in ARMS}

t0 = time.time()
for f in range(5):
    tr = np.where(folds != f)[0]
    va = np.where(folds == f)[0]

    # 13's 1-way encoder, untouched, exactly as row 38 used it.
    Xtr, Xva, Xte = build(X, y, tr, va, X_test)

    # Train and test levels built together from this fold's edges, so a cell means
    # the same thing on both sides.
    lv, lvt = fold_levels(tr, X_test)

    c_tr, c_va, c_te = encode_pairs(y, tr, va, ALL_PAIRS, lv, lvt)
    Xtr = pd.concat([Xtr, c_tr], axis=1)
    Xva = pd.concat([Xva, c_va], axis=1)
    Xte = pd.concat([Xte, c_te], axis=1)
    del c_tr, c_va, c_te
    gc.collect()

    for a in ARMS:
        cols = arm_cols(a, Xtr)
        m = make(N_EST)
        ts = time.time()
        m.fit(Xtr[cols], y[tr])
        p = m.predict_proba(Xva[cols])[:, 1]
        oof[a][va] = p
        test_pred[a] += m.predict_proba(Xte[cols])[:, 1] / 5
        per_fold[a].append(float(roc_auc_score(y[va], p)))
        note(f"  fold {f} {a:>6} ({len(cols):>3} feat): {per_fold[a][-1]:.6f}  "
             f"({hhmm(time.time() - ts)})")
        del m
        gc.collect()

    del Xtr, Xva, Xte
    gc.collect()
    done = time.time() - t0
    note(f"fold {f} done, elapsed {hhmm(done)}, "
         f"about {hhmm(done / (f + 1) * (4 - f))} left")

cv = {a: float(np.mean(per_fold[a])) for a in ARMS}
sd = {a: float(np.std(per_fold[a])) for a in ARMS}
print()
print(f"{'arm':>8} {'CV':>10} {'fold sd':>10}")
for a in ARMS:
    print(f"{a:>8} {cv[a]:>10.6f} {sd[a]:>10.6f}")
note("run done, " + ", ".join(f"{a}:{cv[a]:.6f}" for a in ARMS))


In [ ]:
repro = cv["base"] - BASELINE_CV
REPRODUCED = abs(repro) < 1e-4
print(f"arm base    : {cv['base']:.6f}")
print(f"ledger row 38: {BASELINE_CV:.6f}")
print(f"difference   : {repro:+.2e}   "
      f"{'inside' if REPRODUCED else 'OUTSIDE'} the 1e-04 tolerance")
if SMOKE:
    print("SMOKE: subsampled, so this is EXPECTED to be far out and is not a check.")

base = np.array(per_fold["base"])
rows = []
for a in ARMS:
    if a == "base":
        continue
    d = np.array(per_fold[a]) - base
    rows.append((a, d.mean(), d.std(ddof=1), int((d > 0).sum()), d))

print()
print(f"{'arm':>8} {'paired mean':>13} {'paired sd':>11} {'wins':>7} {'mean/sd':>9}")
for a, m, sdv, w, _ in rows:
    print(f"{a:>8} {m:>+13.6f} {sdv:>11.6f} {w:>5}/5 "
          f"{(m / sdv if sdv else float('nan')):>9.1f}")
print()
for a, m, sdv, w, d in rows:
    print(f"  {a:>6}: per-fold {np.round(d, 6).tolist()}")

# top9's pairs were chosen on fold 0, so fold 0 is the one fold where its CV is
# optimistically biased. Folds 1 to 4 are where the selection never looked, and the
# gap between the two is the size of the bias.
d9 = np.array(per_fold["top9"]) - base
print()
print(f"top9, all five folds     : {d9.mean():+.6f}")
print(f"top9, folds 1 to 4 only  : {d9[1:].mean():+.6f}   "
      "(selection never saw these)")
print(f"selection bias, estimated: {d9.mean() - d9[1:].mean():+.6f}")
print("all66 involves no selection, so it needs no such correction.")

blocked = None
if not (LEAK_OK and CLEAN):
    blocked = "a 1-way leak check failed"
elif not XCLEAN:
    blocked = "a crossed leak check failed"
elif not ENCODER_MATCH:
    blocked = "the 1-way encoder does not match 13"
elif not SMOKE and not ALIGNED:
    blocked = "fold alignment failed"
elif not SMOKE and not REPRODUCED:
    blocked = f"the base arm missed row 38 by {repro:+.2e}"

print()
if blocked:
    print(f"VERDICT: blocked, {blocked}")
elif SMOKE:
    print("SMOKE: no verdict, subsampled rows cannot resolve differences this small.")
else:
    a, m, sdv, w, _ = max(rows, key=lambda r: r[1])
    sig = w >= 4 and sdv > 0 and m > 2 * sdv
    if sig and m >= GATE_FLOOR:
        print(f"VERDICT: carry to a second seed. {a} clears the bar at {m:+.6f},")
        print("  and one seed does not make an improvement in this repo (rows 26, 38).")
    elif sig:
        print(f"VERDICT: parity. {a} is paired-significant at {m:+.6f} but under the")
        print(f"  {GATE_FLOOR:.5f} floor. Logged as parity, not as an improvement.")
    else:
        print("VERDICT: null. The interactions 31 measured are real and the trees")
        print("  already had them, which is row 19's result one dimension up.")


In [ ]:
pre = "SMOKE_" if SMOKE else ""
for a in ARMS:
    np.save(OUT / f"{pre}xgb_pair_{a}_oof.npy", oof[a])
    np.save(OUT / f"{pre}xgb_pair_{a}_test.npy", test_pred[a])
print(f"wrote {pre}xgb_pair_<arm>_oof.npy and _test.npy for {ARMS}")
print()
print("ledger lines, one per arm:")
for a in ARMS:
    print(f"  xgb_pair_{a:<6}  cv_mean {cv[a]:.6f}  cv_std {sd[a]:.6f}")
print()
print(f"  1-way leak checks {'PASS' if CLEAN else 'FAILED'}, "
      f"crossed leak checks {'PASS' if XCLEAN else 'FAILED'}, "
      f"fold alignment {'verified' if ALIGNED else 'NOT verified'}, "
      f"encoder {'matches 13' if ENCODER_MATCH else 'DIFFERS'}")
